# SDXL Generator Attribution Test Set (p01–p50)


## Phase 1: Install + download weights

In [1]:
!pip install -U diffusers transformers accelerate safetensors huggingface_hub


Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Downloading from mainland China: keep the mirror.
# Running this notebook outside mainland China instead: comment the next line out.
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
# Put this on the DATA disk (not the small system disk), e.g. /root/autodl-tmp
CACHE_DIR = "/root/autodl-tmp/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print("HF_ENDPOINT =", os.environ["HF_ENDPOINT"])
print("CACHE_DIR   =", CACHE_DIR)


HF_ENDPOINT = https://hf-mirror.com
CACHE_DIR   = /root/autodl-tmp/hf_cache


In [3]:
from huggingface_hub import snapshot_download

# Downloads the fp16 weights only (skips the larger fp32 files) — no GPU required.
snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=CACHE_DIR,
    allow_patterns=["*.json", "*.txt", "*fp16*", "*.model"],
)
print("Download complete.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Download complete.


Once this finishes, check the folder size (`!du -sh /root/autodl-tmp/hf_cache`) and confirm it looks like several GB.

**Now shut down this no-GPU instance and switch to your RTX 3090 instance.** Re-open this notebook there — the weights are on the data disk, so Phase 2 will load them from `CACHE_DIR` instead of re-downloading.

## Phase 2 : Generate the 50 images

In [4]:
import csv
import time
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
from datetime import date

import torch
from diffusers import StableDiffusionXLPipeline


In [5]:
OUT_DIR = "/root/autodl-tmp/out/sdxl"
META = "/root/autodl-tmp/out/sdxl_meta.csv"
STEPS, GUIDANCE, SIZE = 30, 5.0, 1024
LIMIT = 50

os.makedirs(OUT_DIR, exist_ok=True)


In [6]:
PROMPTS = [
    ('p01', 'portrait', 'A middle-aged woman with short gray hair smiling in a sunlit kitchen.'),
    ('p02', 'portrait', 'A young man wearing a denim jacket standing at a bus stop on a rainy evening.'),
    ('p03', 'portrait', 'An elderly fisherman mending a net on a wooden dock.'),
    ('p04', 'portrait', 'A little girl in a yellow raincoat jumping in a puddle.'),
    ('p05', 'portrait', 'A teenage boy playing a guitar on a bedroom floor.'),
    ('p06', 'group', 'Three friends laughing around a table at an outdoor café.'),
    ('p07', 'group', 'A group of children playing football on a dusty field.'),
    ('p08', 'group', 'Commuters waiting on a crowded subway platform.'),
    ('p09', 'group', 'A family having a picnic in a park on a sunny afternoon.'),
    ('p10', 'group', 'Two cooks working side by side in a busy restaurant kitchen.'),
    ('p11', 'landscape', 'A misty mountain valley at sunrise with a small river.'),
    ('p12', 'landscape', 'A tropical beach with palm trees and a wooden boat.'),
    ('p13', 'landscape', 'A snowy pine forest with a cabin and smoke rising from the chimney.'),
    ('p14', 'landscape', 'A rice terrace on a hillside after the rain.'),
    ('p15', 'landscape', 'A desert road stretching toward distant mountains at dusk.'),
    ('p16', 'animal', 'A tabby cat sleeping on a windowsill.'),
    ('p17', 'animal', 'A golden retriever running along a shoreline.'),
    ('p18', 'animal', 'A flock of birds resting on a power line at sunset.'),
    ('p19', 'animal', 'A brown horse standing in a grassy field.'),
    ('p20', 'animal', 'A butterfly landing on a purple flower.'),
    ('p21', 'food', 'A bowl of ramen with a soft-boiled egg and green onions.'),
    ('p22', 'food', 'A slice of chocolate cake on a white plate.'),
    ('p23', 'food', 'A street vendor grilling skewers at night.'),
    ('p24', 'food', 'A fresh fruit platter with watermelon, mango, and grapes.'),
    ('p25', 'food', 'A plate of nasi lemak with sambal, egg, and fried anchovies.'),
    ('p26', 'indoor', 'A cozy living room with a sofa, bookshelf, and a floor lamp.'),
    ('p27', 'indoor', 'A modern office with rows of desks and large windows.'),
    ('p28', 'indoor', 'A cluttered workshop with tools hanging on the wall.'),
    ('p29', 'indoor', 'A small bedroom with a single bed and morning light.'),
    ('p30', 'indoor', 'A library reading room with tall shelves.'),
    ('p31', 'street', 'A narrow alley in an old town with hanging lanterns.'),
    ('p32', 'street', 'A busy intersection in a big city at night.'),
    ('p33', 'street', 'A quiet suburban street lined with trees in autumn.'),
    ('p34', 'street', 'A market street full of stalls and shoppers.'),
    ('p35', 'street', 'A train station platform in the early morning.'),
    ('p36', 'object', 'A wristwatch lying on a wooden table.'),
    ('p37', 'object', 'A pair of worn running shoes on a concrete floor.'),
    ('p38', 'object', 'A cup of coffee next to an open notebook and a pen.'),
    ('p39', 'object', 'A bicycle leaning against a brick wall.'),
    ('p40', 'object', 'A vase of sunflowers on a kitchen counter.'),
    ('p41', 'illustration', 'A watercolor painting of a lighthouse on a cliff.'),
    ('p42', 'illustration', 'A flat vector illustration of a city skyline at sunset.'),
    ('p43', 'illustration', 'A pencil sketch of an old man reading a newspaper.'),
    ('p44', 'illustration', 'An anime-style illustration of a girl walking through a cherry blossom lane.'),
    ('p45', 'illustration', "A children's book illustration of a bear having tea in a forest."),
    ('p46', 'text', 'A storefront sign that reads "OPEN 24 HOURS" above a small shop door.'),
    ('p47', 'text', 'A chalkboard menu listing three coffee drinks and their prices.'),
    ('p48', 'text', 'A poster on a wall that says "Welcome to the Science Fair".'),
    ('p49', 'text', 'A street name sign reading "Maple Street" on a pole.'),
    ('p50', 'text', 'A birthday card with the words "Happy Birthday" on the front.'),
]

HEADER = [
    "id", "category", "prompt", "generator", "model_id", "seed", "steps",
    "guidance", "width", "height", "format", "invisible_watermark", "date", "seconds",
]
prompts = PROMPTS[:LIMIT] if LIMIT else PROMPTS
print(f"{len(prompts)} prompts queued (p01 to {prompts[-1][0]})")


50 prompts queued (p01 to p50)


In [7]:
   pipe = StableDiffusionXLPipeline.from_pretrained(
       MODEL_ID,
       cache_dir=CACHE_DIR,
       torch_dtype=torch.float16,
       variant="fp16",
       use_safetensors=True,
       add_watermarker=False,
   )
   pipe.enable_model_cpu_offload()
   pipe.enable_attention_slicing()

/root/miniconda3/lib/python3.12/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [8]:
new_file = not os.path.exists(META)
with open(META, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if new_file:
        writer.writerow(HEADER)

    for idx, (pid, cat, prompt) in enumerate(prompts, 1):
        path = f"{OUT_DIR}/{pid}.png"
        if os.path.exists(path):
            print(f"[{idx}/{len(prompts)}] {pid} already exists, skipping")
            continue

        seed = idx  # fixed seed per prompt id for reproducibility
        gen = torch.Generator("cuda").manual_seed(seed)
        t0 = time.time()
        image = pipe(
            prompt,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            height=SIZE,
            width=SIZE,
            generator=gen,
        ).images[0]
        image.save(path)

        writer.writerow([
            pid, cat, prompt, "sdxl", MODEL_ID, seed, STEPS, GUIDANCE,
            SIZE, SIZE, "png", "no", date.today().isoformat(),
            round(time.time() - t0, 1),
        ])
        f.flush()
        print(f"[{idx}/{len(prompts)}] {pid} done ({time.time()-t0:.1f}s)")

print("All done.")


[1/50] p01 already exists, skipping
[2/50] p02 already exists, skipping
[3/50] p03 already exists, skipping
[4/50] p04 already exists, skipping
[5/50] p05 already exists, skipping
[6/50] p06 already exists, skipping
[7/50] p07 already exists, skipping
[8/50] p08 already exists, skipping
[9/50] p09 already exists, skipping
[10/50] p10 already exists, skipping
[11/50] p11 already exists, skipping
[12/50] p12 already exists, skipping
[13/50] p13 already exists, skipping
[14/50] p14 already exists, skipping
[15/50] p15 already exists, skipping
[16/50] p16 already exists, skipping
[17/50] p17 already exists, skipping
[18/50] p18 already exists, skipping
[19/50] p19 already exists, skipping
[20/50] p20 already exists, skipping
[21/50] p21 already exists, skipping
[22/50] p22 already exists, skipping
[23/50] p23 already exists, skipping
[24/50] p24 already exists, skipping
[25/50] p25 already exists, skipping
[26/50] p26 already exists, skipping
[27/50] p27 already exists, skipping
[28/50] p2